# Clustering, full model, and ten-fold evaluation

Fixed UNI-v2 / PCA128 / K90 / P50 candidate. Each internal fold learns its own representation and Cox coefficients from training patients only. External predictions use the full development model.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd() if (Path.cwd() / 'helpers').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from helpers.io import settings, destination
cfg = settings()
print('Mode:', cfg['mode'])


Optional demo mode creates independent synthetic patients and 1536-dimensional patch vectors. These are unrelated to the TCGA image example. Set `ESCC_MODE=demo` before opening Jupyter to enable it.

In [ ]:
from helpers import demo, training
from helpers.io import inputs
if cfg['mode'] == 'demo':
    demo.generate(cfg)
clinical = inputs(cfg)
clinical.groupby(['cohort', 'image_included', 'treatment']).size()

Patient-balanced patch sampling → L2 → randomized PCA → MiniBatchKMeans → slide-smoothed composition → equal-slide patient average → centered CLR → ridge Cox. The reference is selected from training mean proportions (C15 in the original full-data model).

In [ ]:
oof = training.run(cfg)
oof.groupby(['fold', 'group']).size().unstack(fill_value=0)

The PFS-trained score is βT + γᵀz. B: predicted HR(z)<1; NB: HR(z)≥1. OS is evaluated without fitting a separate OS model. No candidate or threshold selection is rerun.